# Nemotron Reasoning Challenge — Submission

Packages the LoRA adapter for competition scoring.

**How to use:**
1. Upload your trained adapter as a **Kaggle Dataset** (private)
2. Attach it to this notebook via **Add Input** on the right sidebar
3. Update `ADAPTER_INPUT` below to match the dataset path
4. Run all cells → `submission.zip` is produced

**Identity LoRA** (pipeline validation): Run Cell 2 instead of Cell 1 to generate an untrained adapter and submit it. This verifies the submission pipeline works before investing GPU hours in training.

In [ ]:
# -- Cell 1: Copy trained adapter from attached dataset ---------------
# Update this path to match your uploaded Kaggle Dataset name
ADAPTER_INPUT = "/kaggle/input/nemotron-lora-v1"

import shutil, os
from pathlib import Path

OUT = Path("/kaggle/working")

# Find adapter_config.json in the input dataset
adapter_config = None
for root, dirs, files in os.walk(ADAPTER_INPUT):
    if "adapter_config.json" in files:
        adapter_config = root
        break

if adapter_config is None:
    raise FileNotFoundError(
        f"No adapter_config.json found under {ADAPTER_INPUT}. "
        f"Contents: {os.listdir(ADAPTER_INPUT)}"
    )

# Copy adapter files to working directory
for f in os.listdir(adapter_config):
    src = os.path.join(adapter_config, f)
    if os.path.isfile(src):
        shutil.copy2(src, OUT / f)

print(f"Adapter files copied from: {adapter_config}")
print(f"Files in /kaggle/working: {os.listdir(OUT)}")
assert (OUT / "adapter_config.json").exists(), "adapter_config.json missing!"

In [ ]:
# -- Cell 2: Identity LoRA (pipeline validation only) -----------------
# SKIP this cell if you already ran Cell 1 with a trained adapter.
# This creates an untrained adapter to verify the submission pipeline.

import kagglehub
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)

lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.save_pretrained("/kaggle/working")
print("Identity LoRA saved to /kaggle/working")

In [ ]:
# -- Cell 3: Package submission.zip -----------------------------------
import subprocess, os

os.chdir("/kaggle/working")
subprocess.run("zip -m submission.zip *", shell=True, check=True)

size_mb = os.path.getsize("/kaggle/working/submission.zip") / 1e6
print(f"submission.zip created: {size_mb:.1f} MB")